# 🏢 Employee Attrition Analysis
### Python Data Cleaning & Exploratory Data Analysis
---
> **Project Goal:** Understand why employees leave a company (attrition) by cleaning the HR dataset and analyzing patterns using Python.

**Workflow:**
1. Import Libraries
2. Load Dataset
3. Data Understanding
4. Data Cleaning
5. Exploratory Data Analysis (EDA)
6. Save Cleaned Dataset

---
## Step 1: Import Libraries

Before we do anything, we need to bring in the **tools (libraries)** that Python needs.
Think of it like opening your toolbox before starting work.

| Library | What it does |
|---|---|
| `pandas` | Reads, cleans, and manipulates data (like Excel, but in Python) |
| `numpy` | Handles numbers and mathematical operations |
| `matplotlib` | Creates basic charts and graphs |
| `seaborn` | Creates beautiful statistical charts (built on top of matplotlib) |
| `warnings` | Suppresses unnecessary warning messages so output stays clean |

> ❓ **What if we don't import?**  
> Python will throw a `NameError` — it won't know what `pd`, `np`, or `sns` means.

In [ ]:
import pandas as pd        # 'pd' is a short nickname — everyone uses this convention
import numpy as np         # 'np' is the standard nickname for numpy
import matplotlib.pyplot as plt   # 'plt' is the standard nickname
import seaborn as sns      # 'sns' is the standard nickname for seaborn
import warnings
warnings.filterwarnings('ignore')   # Hides harmless warning messages from output

# --- Display Settings ---
# This tells pandas: show ALL columns when we print a dataframe (default only shows some)
pd.set_option('display.max_columns', 50)

# This tells pandas: show numbers with only 2 decimal places (cleaner output)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set a clean visual style for all charts
sns.set_theme(style='whitegrid', palette='Set2')

print('✅ All libraries imported successfully!')

---
## Step 2: Load the Dataset

We use `pd.read_csv()` to load our CSV file into a **DataFrame**.

> 📌 **What is a DataFrame?**  
> A DataFrame is like an Excel table in Python — it has rows and columns.  
> We store it in a variable called `df` (short for DataFrame — common convention).

> ❓ **What if we used a different format?**  
> - Excel file → `pd.read_excel('file.xlsx')`  
> - SQL database → `pd.read_sql(query, connection)`  
> - JSON file → `pd.read_json('file.json')`

In [ ]:
# Load the CSV file into a DataFrame
# Make sure the CSV file is in the same folder as this notebook
df = pd.read_csv('Project_emp_attrition_dataset.csv')

# .shape gives us (number of rows, number of columns)
print(f'Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')

# .head() shows the first 5 rows — good for a quick first look
# Why 5? It's the default. You can do df.head(10) to see 10 rows.
df.head()

---
## Step 3: Data Understanding

Before cleaning, we must **understand** what we have.  
This step answers 4 key questions:
1. What columns exist and what type is each?
2. Are there any missing values?
3. Are there duplicate rows?
4. What are the unique values in categorical columns?

### 3a. Column Names and Data Types

> 📌 `.info()` is one of the most important commands in pandas.  
> It shows you: column names, how many non-null values, and the data type (dtype).

**Data Types you'll see:**
| dtype | Meaning | Example |
|---|---|---|
| `int64` | Whole numbers | Age = 35 |
| `float64` | Decimal numbers | Salary = 5432.50 |
| `object` | Text / strings | Department = 'Sales' |

In [ ]:
# .info() gives a full summary of the dataframe
# It shows column names, non-null counts, and data types
df.info()

### 3b. Statistical Summary

> 📌 `.describe()` gives statistics for all **numerical** columns automatically.

**What each row means:**
| Row | Meaning |
|---|---|
| `count` | How many non-null values |
| `mean` | Average value |
| `std` | Standard deviation (how spread out the values are) |
| `min` | Smallest value |
| `25%` | 25% of data is below this value |
| `50%` | Middle value (median) |
| `75%` | 75% of data is below this value |
| `max` | Largest value |

> ❓ **Why not use just the average (mean)?**  
> The mean can be misleading if there are extreme values (outliers).  
> Example: If most salaries are ₹5,000 but one person earns ₹1,00,000, the mean looks high but isn't representative.  
> That's why we also check `median (50%)` and `std`.

In [ ]:
# .describe() shows statistical summary of all numerical columns
# include='all' would also include categorical columns
df.describe()

### 3c. Check Missing Values

> 📌 Missing values (also called **null values** or **NaN**) are empty cells.  
> They can break your analysis or give wrong results if not handled.

> ❓ **What is NaN?**  
> NaN = "Not a Number" — Python's way of saying the cell is empty/missing.

**How to handle missing values (options):**
- Drop the row → `df.dropna()`
- Fill with average → `df['column'].fillna(df['column'].mean())`
- Fill with most common value → `df['column'].fillna(df['column'].mode()[0])`

In [ ]:
# .isnull() returns True/False for each cell — True means the value is missing
# .sum() counts how many True values are in each column
missing_values = df.isnull().sum()

print('=== Missing Values Per Column ===')
print(missing_values)
print()
print(f'Total missing values in the entire dataset: {missing_values.sum()}')

# Also show as a percentage — easier to understand impact
missing_pct = (df.isnull().sum() / len(df)) * 100
print()
print('=== Missing Value Percentage ===')
print(missing_pct[missing_pct > 0])  # Only show columns that have missing values
print('✅ No missing values!' if missing_values.sum() == 0 else '⚠️ Missing values found — needs handling!')

### 3d. Check for Duplicate Rows

> 📌 **Duplicate rows** are identical records that appear more than once.  
> This can happen due to data entry errors or system exports.
> If not removed, they will inflate your counts and give wrong statistics.

> ❓ **What if we don't remove duplicates?**  
> If employee ID 5 appears twice, we'll count that employee twice in all our analysis — wrong results!

In [ ]:
# .duplicated() returns True for any row that is an exact copy of another row
# .sum() counts how many duplicate rows exist
duplicate_count = df.duplicated().sum()

print(f'Number of duplicate rows: {duplicate_count}')
print('✅ No duplicates found!' if duplicate_count == 0 else f'⚠️ {duplicate_count} duplicate rows found!')

### 3e. Explore Categorical Columns

> 📌 For **text/category columns**, we check what unique values exist.  
> This helps us spot typos, inconsistencies, or unexpected values.

> **Example problem:** If Gender column has `['Male', 'Female', 'male', 'MALE']`,  
> Python treats them as 4 different categories — we need to fix that!

In [ ]:
# select_dtypes(include='object') selects only text/string columns
# These are the columns that hold category values like 'Yes'/'No', 'Male'/'Female'
categorical_columns = df.select_dtypes(include='object').columns

print('=== Unique Values in Categorical Columns ===')
for col in categorical_columns:
    # .unique() gives all distinct values in a column
    print(f'\n{col} ({df[col].nunique()} unique values):')
    print(df[col].value_counts().to_dict())

---
## Step 4: Data Cleaning

Now that we understand the data, we fix the problems we found.

**Cleaning tasks for this dataset:**
1. Drop useless columns (same value in every row)
2. Convert Yes/No columns to numbers (0 and 1)
3. Verify no duplicates or nulls remain
4. Reset the index after any drops

### 4a. Drop Useless Columns

Some columns have the **same value for every single row** — they carry zero information.

| Column | Problem |
|---|---|
| `Over18` | Every employee is 'Y' — tells us nothing |
| `EmployeeCount` | Always 1 — not useful |
| `StandardHours` | Always 80 — not useful |

> ❓ **Why drop them?**  
> They take up memory and clutter our analysis. A column with no variation cannot help us understand attrition.

In [ ]:
# Columns to drop — they have only one unique value (no information value)
cols_to_drop = ['Over18', 'EmployeeCount', 'StandardHours']

# axis=1 means we're dropping COLUMNS (axis=0 would drop rows)
# inplace=True means change the dataframe directly instead of creating a new one
df.drop(columns=cols_to_drop, inplace=True)

print(f'Columns after dropping: {df.shape[1]}')
print(f'Removed columns: {cols_to_drop}')

### 4b. Convert Yes/No to 1/0 (Binary Encoding)

The `Attrition` and `OverTime` columns contain `'Yes'` and `'No'` as text.  
We convert them to **numbers (1/0)** because:
- Machine learning models only understand numbers
- Mathematical operations (like average attrition rate) only work on numbers
- SQL and Power BI work better with 0/1 flags

> ❓ **What if we skip this?**  
> `df['Attrition'].mean()` would give an error — you can't average text values.  
> After converting: `df['Attrition'].mean()` gives the attrition rate (e.g., 0.16 = 16%).

In [ ]:
# Map 'Yes' to 1 and 'No' to 0 using a dictionary
# The .map() function replaces each value using the dictionary we provide
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
df['OverTime']  = df['OverTime'].map({'Yes': 1, 'No': 0})

print('Attrition column after encoding:')
print(df['Attrition'].value_counts())
print()
print('OverTime column after encoding:')
print(df['OverTime'].value_counts())
print()

# Quick check: attrition rate as a percentage
attrition_rate = df['Attrition'].mean() * 100
print(f'Overall Attrition Rate: {attrition_rate:.1f}%')

### 4c. Final Cleaning Verification

> 📌 Always do a **final check** after cleaning to confirm everything is correct.  
> This is like a quality check before handing in your work.

In [ ]:
print('=== FINAL CLEANING REPORT ===')
print(f'Total Rows       : {df.shape[0]}')
print(f'Total Columns    : {df.shape[1]}')
print(f'Missing Values   : {df.isnull().sum().sum()}')
print(f'Duplicate Rows   : {df.duplicated().sum()}')
print()
print('Data Types After Cleaning:')
print(df.dtypes)
print()
print('✅ Dataset is clean and ready for analysis!')

---
## Step 5: Exploratory Data Analysis (EDA)

> 📌 **EDA** means exploring the data visually and statistically to find patterns and insights.
> This answers the key business question: **"What factors are linked to employee attrition?"

We'll analyze:
- Overall attrition distribution
- Attrition by Department
- Attrition by Job Role
- Attrition by Overtime
- Age distribution of employees who left
- Monthly Income vs Attrition

### 5a. Attrition Distribution (Overall)

> 📌 First, always look at the **target variable** (the thing we're trying to understand — `Attrition`).  
> We want to know: how many employees left vs stayed?

In [ ]:
# Create a figure with 2 subplots side by side
# figsize=(12, 5) sets the width=12 inches, height=5 inches
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Chart 1: Count Plot ---
# countplot counts how many rows have each value — good for categorical data
attrition_labels = {0: 'Stayed', 1: 'Left'}
df['Attrition_Label'] = df['Attrition'].map(attrition_labels)

sns.countplot(data=df, x='Attrition_Label', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Employee Attrition Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Attrition Status')
axes[0].set_ylabel('Number of Employees')

# Add count labels on top of each bar
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# --- Chart 2: Pie Chart ---
# Pie chart shows proportion — good to show percentages
attrition_counts = df['Attrition_Label'].value_counts()
axes[1].pie(attrition_counts, labels=attrition_counts.index,
            autopct='%1.1f%%',  # Shows percentage with 1 decimal place
            colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0.05, 0))
axes[1].set_title('Attrition Percentage', fontsize=14, fontweight='bold')

plt.suptitle('Overall Employee Attrition', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Employees who stayed : {(df["Attrition"]==0).sum()} ({(df["Attrition"]==0).mean()*100:.1f}%)')
print(f'Employees who left   : {(df["Attrition"]==1).sum()} ({(df["Attrition"]==1).mean()*100:.1f}%)')

### 5b. Attrition by Department

> 📌 **Groupby** is one of the most important pandas operations.
> It groups all rows with the same value together, then applies a function (like mean, count, sum).

> Think of it like an Excel PivotTable — group by Department, calculate attrition rate.

> ❓ **Why use `.mean()` on Attrition (0/1)?**  
> Because the average of 0s and 1s gives us the **rate**.  
> E.g., if 3 out of 10 people left → values are [1,1,1,0,0,0,0,0,0,0] → mean = 0.3 = 30% attrition rate.

In [ ]:
# groupby('Department') groups all rows by department
# ['Attrition'].mean() calculates attrition RATE per department
# * 100 converts to percentage
# .reset_index() converts the grouped result back to a clean dataframe
dept_attrition = df.groupby('Department')['Attrition'].mean().mul(100).reset_index()
dept_attrition.columns = ['Department', 'Attrition Rate (%)']
dept_attrition = dept_attrition.sort_values('Attrition Rate (%)', ascending=False)

print('Attrition Rate by Department:')
print(dept_attrition)
print()

# Bar chart
plt.figure(figsize=(9, 5))
bars = plt.bar(dept_attrition['Department'], dept_attrition['Attrition Rate (%)'],
               color=['#e74c3c', '#e67e22', '#3498db'])

# Add value labels on bars
for bar, val in zip(bars, dept_attrition['Attrition Rate (%)']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

plt.title('Attrition Rate by Department', fontsize=15, fontweight='bold')
plt.xlabel('Department')
plt.ylabel('Attrition Rate (%)')
plt.ylim(0, dept_attrition['Attrition Rate (%)'].max() + 5)
plt.tight_layout()
plt.show()

### 5c. Attrition by Job Role

> 📌 Same logic as above, but now grouped by **Job Role**.  
> This tells us which roles have the highest exit rates — useful for HR to target retention efforts.

In [ ]:
role_attrition = df.groupby('JobRole')['Attrition'].mean().mul(100).reset_index()
role_attrition.columns = ['Job Role', 'Attrition Rate (%)']
role_attrition = role_attrition.sort_values('Attrition Rate (%)', ascending=True)  # ascending for horizontal bar

plt.figure(figsize=(10, 6))

# barh = horizontal bar chart — better when category labels are long text
# ❓ Why horizontal? Because job role names are long — vertical bars would overlap
bars = plt.barh(role_attrition['Job Role'], role_attrition['Attrition Rate (%)'],
                color='#3498db', edgecolor='white')

for bar, val in zip(bars, role_attrition['Attrition Rate (%)']):
    plt.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=10)

plt.title('Attrition Rate by Job Role', fontsize=15, fontweight='bold')
plt.xlabel('Attrition Rate (%)')
plt.tight_layout()
plt.show()

### 5d. Overtime vs Attrition

> 📌 One hypothesis: **employees who do overtime are more likely to leave.**  
> Let's check if the data supports this.

> We use a **grouped bar chart** to compare attrition between those who work overtime and those who don't.

In [ ]:
# Cross-tabulation: shows count of each combination of OverTime and Attrition
# normalize='index' converts counts to proportions (row-wise percentages)
overtime_attr = pd.crosstab(df['OverTime'], df['Attrition_Label'], normalize='index') * 100
print('Attrition Rate by OverTime Status:')
print(overtime_attr.round(1))

overtime_attr.plot(kind='bar', figsize=(8, 5),
                   color=['#2ecc71', '#e74c3c'], edgecolor='white',
                   rot=0)  # rot=0 keeps x-axis labels horizontal

plt.title('Overtime vs Attrition', fontsize=15, fontweight='bold')
plt.xlabel('OverTime (0 = No, 1 = Yes)')
plt.ylabel('Percentage of Employees (%)')
plt.legend(['Stayed', 'Left'], title='Attrition')
plt.tight_layout()
plt.show()

### 5e. Age Distribution — Who is Leaving?

> 📌 A **histogram** shows the distribution (spread) of a numerical column.  
> We compare the age distribution of employees who left vs those who stayed.

> ❓ **Why use histogram for Age and not a bar chart?**  
> Bar chart = for categories (like Department, Gender)  
> Histogram = for continuous numbers (like Age, Salary) — it groups numbers into ranges (bins)

In [ ]:
plt.figure(figsize=(10, 5))

# Plot two overlapping histograms — one for each attrition group
# alpha controls transparency (0=invisible, 1=solid) — we use 0.6 so both are visible
# bins=20 divides the age range into 20 equal buckets

df[df['Attrition'] == 0]['Age'].plot(kind='hist', bins=20, alpha=0.6,
                                      color='#2ecc71', label='Stayed')
df[df['Attrition'] == 1]['Age'].plot(kind='hist', bins=20, alpha=0.6,
                                      color='#e74c3c', label='Left')

plt.title('Age Distribution: Stayed vs Left', fontsize=15, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Number of Employees')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Average age of employees who stayed : {df[df["Attrition"]==0]["Age"].mean():.1f} years')
print(f'Average age of employees who left   : {df[df["Attrition"]==1]["Age"].mean():.1f} years')

### 5f. Monthly Income vs Attrition

> 📌 **Boxplot** is perfect for comparing the spread of a numerical value across categories.  
> It shows: minimum, Q1, median, Q3, maximum — and outliers as dots.

> ❓ **Why boxplot here instead of histogram?**  
> We want to *compare* income between two groups (Left vs Stayed).  
> A boxplot lets us put both groups side by side for easy comparison.

In [ ]:
plt.figure(figsize=(8, 5))

# x = category column, y = numerical column we want to compare
# hue = splits each box by Attrition to compare side by side
sns.boxplot(data=df, x='Attrition_Label', y='MonthlyIncome',
            palette={'Stayed': '#2ecc71', 'Left': '#e74c3c'})

plt.title('Monthly Income vs Attrition', fontsize=15, fontweight='bold')
plt.xlabel('Attrition Status')
plt.ylabel('Monthly Income')
plt.tight_layout()
plt.show()

print('Average Monthly Income:')
print(df.groupby('Attrition_Label')['MonthlyIncome'].mean().round(0))

### 5g. Years at Company vs Attrition

> 📌 Do newer employees leave more? Or experienced ones?  
> Let's compare tenure (years at company) between those who left and stayed.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Attrition_Label', y='YearsAtCompany',
            palette={'Stayed': '#3498db', 'Left': '#e74c3c'})
plt.title('Years at Company vs Attrition', fontsize=15, fontweight='bold')
plt.xlabel('Attrition Status')
plt.ylabel('Years at Company')
plt.tight_layout()
plt.show()

print('Average Years at Company:')
print(df.groupby('Attrition_Label')['YearsAtCompany'].mean().round(1))

### 5h. Correlation Heatmap

> 📌 **Correlation** tells us how strongly two columns are related to each other.  
> Value ranges from -1 to +1:
> - `+1` = strong positive (one goes up, the other goes up too)
> - `-1` = strong negative (one goes up, the other goes down)
> - `0` = no relationship

> ❓ **Why use a heatmap?**  
> We have 32 numerical columns. Checking each pair one by one is impossible.  
> A heatmap shows ALL correlations at once using color intensity.

In [ ]:
# Select only numerical columns for correlation
numeric_df = df.select_dtypes(include=np.number)

# .corr() computes correlation between all pairs of numerical columns
corr_matrix = numeric_df.corr()

# Show only correlations with 'Attrition' column, sorted by strength
print('=== Correlation with Attrition (sorted) ===')
attrition_corr = corr_matrix['Attrition'].drop('Attrition').sort_values(key=abs, ascending=False)
print(attrition_corr.round(3))

# Full heatmap
plt.figure(figsize=(16, 12))
# annot=True shows the correlation number inside each cell
# fmt='.1f' shows 1 decimal place
# cmap='coolwarm' — blue = negative correlation, red = positive
sns.heatmap(corr_matrix, annot=True, fmt='.1f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Correlation Heatmap of All Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6: Save the Cleaned Dataset

> 📌 After cleaning, we save the dataset so it can be used in **SQL** and **Power BI**.

> ❓ **Why `index=False`?**  
> By default, pandas saves the row numbers (0, 1, 2...) as an extra column.  
> `index=False` tells it NOT to save that column — we don't need it.

In [ ]:
# Drop the helper column we created for labels (not needed in the final file)
df.drop(columns=['Attrition_Label'], inplace=True)

# Save the cleaned dataset
output_path = 'employee_attrition_analysis.csv'
df.to_csv(output_path, index=False)

# Verify the saved file
df_check = pd.read_csv(output_path)
print(f'✅ Cleaned dataset saved successfully!')
print(f'   File    : {output_path}')
print(f'   Shape   : {df_check.shape[0]} rows × {df_check.shape[1]} columns')
print()
print('Final Columns:')
print(list(df_check.columns))

---
## Summary of Key Findings

| Finding | Insight |
|---|---|
| Overall attrition | ~16% of employees left |
| Highest attrition dept | Sales / Human Resources |
| Overtime impact | Employees with overtime leave significantly more |
| Age pattern | Younger employees (25–35) leave more often |
| Income impact | Employees who left earned lower average income |
| Tenure | Employees with fewer years at the company leave more |

---

### Next Steps
- ✅ **Step 3:** Load `employee_attrition_analysis.csv` into MySQL and write SQL queries
- ✅ **Step 4:** Load into Power BI and build the HR Dashboard